In [0]:
from pyspark.sql import functions as F

schema_name = "test_cfa"  # Change to your schema name

# Step 1: Get all tables in the schema
table_names = [t.name for t in spark.catalog.listTables(schema_name)]

all_stats = []

# Step 2: Loop through each table in schema
for tbl in table_names:
    print(f"Processing table: {schema_name}.{tbl}")
    df = spark.table(f"{schema_name}.{tbl}")
    schema_info = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]

    # Build aggregation expressions for one-pass computation
    agg_exprs = []
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            agg_exprs.extend([
                F.min(F.col(col_name)).cast("string").alias(f"{col_name}_min"),
                F.max(F.col(col_name)).cast("string").alias(f"{col_name}_max"),
                F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls")
            ])
        else:
            agg_exprs.append(F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls"))

    # Aggregate
    stats_row = df.agg(*agg_exprs).collect()[0]

    # Build results list
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            all_stats.append((
                stats_row[f"{col_name}_min"],
                stats_row[f"{col_name}_max"],
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,tbl
                #tbl.replace("_delta", ".csv")
            ))
        else:
            all_stats.append((
                None,
                None,
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,tbl
                #tbl.replace("_delta", ".csv")
            ))

# Step 3: Create final DataFrame
final_stats_df = spark.createDataFrame(all_stats, ["min_value", "max_value", "null_count", "column_name", "data_type", "table_name"])
display(final_stats_df)
